In [1]:
%load_ext autoreload
%autoreload 2
from tasks.dlc_md import DLCMDTask
import os
import torch
import lightning as L
from data import InfoLabel, PrefixSuffixIterable

/home/leog/scratch/sentence_diffusion/venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/leog/scratch/sentence_diffusion/venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may h

In [2]:
torch.set_float32_matmul_precision("medium")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONFAULTHANDLER"] = "1"
os.environ["TORCH_SHOW_CPP_STACKTRACES"] = "1"
os.environ["HYDRA_FULL_ERROR"] = "1"
# os.environ["NCCL_DEBUG"] = "INFO"
# os.environ["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
os.environ["TORCH_NCCL_ASYNC_ERROR_HANDLING"] = "1"
os.environ["TORCH_DISABLE_ADDR2LINE"] = "1"
os.environ["TORCH_FR_BUFFER_SIZE"] = "1024"

In [ ]:
task = DLCMDTask.load_from_checkpoint(
    os.path.join(
        os.environ["LOG_DIR"],
        "checkpoints/",
        "xhv7p1lh",
        "last.ckpt",
    ),
    strict=False,
    map_location=torch.device("cuda"),
)
task.setup()

No sentence-transformers model found with name FacebookAI/roberta-large-mnli. Creating a new one with mean pooling.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/150 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/1016 [00:00<?, ?it/s]

In [70]:
dl = iter(PrefixSuffixIterable.get_dataloader(
    task.val_data,
    batch_size=32,
    prefix_length=task.cfg.prefix_length,
    suffix_length=task.cfg.suffix_length,
    enc_tok=task.encoder.tokenizer if task.encoder is not None else None,
    context_length=task.cfg.context_length,
    dec_tok=task.dit.tokenizer,
    encoder_mode='context' if task.cfg.context_length != 0 else 'suffix',
    encoder_noise=False,
    seed=42,  # Always the same validation set for consistency,
    num_dlc_ph=task.encoder.sem.dlc_len if task.encoder is not None else 0,
))

Token indices sequence length is longer than the specified maximum sequence length for this model (1540 > 1024). Running this sequence through the model will result in indexing errors
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [ ]:
trainer = L.Trainer(
    logger=None,
    accelerator="gpu",
    enable_checkpointing=False,
    precision="bf16-mixed",
    limit_val_batches=4,
    devices=1,
)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
trainer.validate(task)

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/150 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/1016 [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1540 > 1024). Running this sequence through the model will result in indexing errors
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
 23%|██▎       | 58/256 [00:07<00:25,  7.83it/s]

Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/ipykernel/2025a/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [7]:
batch = next(dl)
for k in batch.keys():
    try:
        batch[k] = batch[k].to('cuda')
    except:
        pass
batch = task._fill_DLCs_in_batch(batch)

In [33]:
task.dit.cfg.sampling_mode = "remdm-cap"

In [60]:
task.eval()
with torch.autocast("cuda", dtype=torch.float32):
    with torch.inference_mode():
        prior = batch["input_ids_dec"].clone()
        prior[
            (batch["info_mask_dec"] == InfoLabel.SUFFIX.value)
            + (batch["info_mask_dec"] == InfoLabel.DLC.value)
        ] = task.dit.tokenizer.mask_token_id
        out = task.dit.sample(prior=prior, num_steps=512)

  0%|          | 0/512 [00:00<?, ?it/s]


RuntimeError: indices should be either on cpu or on the same device as the indexed tensor (cpu)

In [40]:
task.dit.tokenizer.batch_decode(batch["input_ids_dec"], skip_special_tokens=False)[1]

' in their 20’s – then I remember I’m definitely in<|think|><|think|> my 40’s.\n11. There are many places in Norway where you can get sparkling (carbonated) water out of the tap. Out of the TAP! Seriously – one faucet will have regular water and the faucet right next to it has wonderful sparkling water. I need this innovation in the parsonage immediately.\n12. One can get used to the little luxuries of being on a reality show cast very quickly: we never had to make any plans or arrange any details of our days. We were told when to wake up, given clothes to wear (I now have a pretty great cold'

In [43]:
task.dit.tokenizer.batch_decode(out, skip_special_tokens=False)[1]

' in their 20’s – then I remember I’m definitely in<|think|><|think|> my early years, too. Oh, I’m thinking about getting a full-time job. None of those now.\nToday, I can’t laugh because I’m in a jumpy mood. I only make one thing up, and then I’ll have to admit that I’m not a comedian. Like, I know, that’s the 1.7 for Hangover.\n“The best thing I’ve done was piss on the TV, huh?”. What a great joke. It suits the excuse that I’m being sarcastic and stupid'

In [54]:
task.eval()

prior = batch["input_ids_dec"].clone()
prior[
    (batch["info_mask_dec"] == InfoLabel.SUFFIX.value)
    + (batch["info_mask_dec"] == InfoLabel.DLC.value)
] = task.dit.tokenizer.mask_token_id
frozen_mask = batch["info_mask_dec"] != InfoLabel.DLC.value
prior = task.dit.sample(
    prior=prior,
    frozen_mask=frozen_mask,
    num_steps=512,
)
#gen_suffix_str_ancestral = task.dit.tokenizer.batch_decode(
#    task.dit.sample(prior=prior, num_steps=256)[:, -task.cfg.suffix_length :]
#)

100%|██████████| 512/512 [00:17<00:00, 29.52it/s]


In [55]:
prior = task.dit.sample(prior=prior, num_steps=256)

100%|██████████| 256/256 [00:03<00:00, 65.31it/s] 


In [56]:
task.dit.tokenizer.batch_decode(prior, skip_special_tokens=False)

[' want to continue to put out good work.\nHHNM: What plans does<|think|><|think|> you have for the upcoming years to follow up?\nHHNM: My plan is to continue to drive the growth efforts on the digital side of things. I am really fortunate to be in the digital area of my career. I want to improve my skills and excel myself in the field. I also want to be a',
 '-lap contest, one where a late scuffle by Sebastian Vettel saw the<|think|><|think|> German hit the wall with a crash of the, hitting the tyres, which was where the car had been in the first lap. It was not a fair fight for Vettel, who is now his fourth of the race, and Vettel did not have the chance to run in his final lap. On lap 29,',
 ' the Colorado School of Public Health, will be the first director of the Center for<|think|><|think|> Behavioral Health at the University of Colorado at Boulder, director of the School of Health, Sciences and Health Sciences at University of Colorado at Boulder as well as the Department of Heal

In [12]:
task.eval()

prefix_str = batch["prefix_str"][:5]
true_suffix_str = batch["suffix_str"][:5]

# Generate conditional on the prefix AND the DLCs
prior = batch["input_ids_dec"][:5].clone()
prior[
    (batch["info_mask_dec"][:5] == InfoLabel.SUFFIX.value)
] = task.dit.tokenizer.mask_token_id
prior = task.dit.sample(prior=prior, num_steps=512)

  0%|          | 0/512 [00:00<?, ?it/s]/home/leog/scratch/huggingface/modules/transformers_modules/kuleshov_hyphen_group/mdlm_hyphen_owt/9e6829bb908d241a074146e4c5c095238bb5e316/modeling_mdlm.py:397: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/home/leog/scratch/huggingface/modules/transformers_modules/kuleshov_hyphen_group/mdlm_hyphen_owt/9e6829bb908d241a074146e4c5c095238bb5e316/modeling_mdlm.py:147: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/home/leog/scratch/huggingface/modules/transformers_modules/kuleshov_hyphen_group/mdlm_hyphen_owt/9e6829bb908d241a074146e4c5c095238bb5e316/modeling_mdlm.py:285: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with t

In [14]:
task.dit.tokenizer.batch_decode(prior, skip_special_tokens=False)

[' want to continue to put out good work.\nHHNM: What plans does<|think|><|think|> you have for the album?\nHHNM: I’m going to work on the album. I’m so excited to put it out with the greatest band in the world. I can’t say that’s my last album. I’m sure it’ll be a',
 '-lap contest, one where a late scuffle by Sebastian Vettel saw the<|think|><|think|> German retire in the middle of the first lap.\nThe German dominated the final lap of the race, after a surprisingly slow start over a second, but it was not until in the session that the Spani Bull launched a stunning finish at the end of the race to win the race at the same time.\nVettel',
 ' the Colorado School of Public Health, will be the first director of the Center for<|think|><|think|> Health Initiatives, a program that assists local leaders to improve their health and education.\nThe program includes education, research and development to develop ideas and practices to improve health and education system quality in the state of C

In [11]:
task.dit.tokenizer.batch_decode(prior, skip_special_tokens=True)

[' want to continue to put out good work.\nHHNM: What plans does',
 '-lap contest, one where a late scuffle by Sebastian Vettel saw the',
 ' the Colorado School of Public Health, will be the first director of the Center for',
 ' exceptional results, as it is superior to residential grade oil based polyurethane',
 ' among the highest rates in the nation. (That figure excludes families in which the',
 ' own unique challenges.\nLarge capital flows are entering the E&S market.',
 ' to wash their hands, use sanitizer, and wipe down equipment frequently.',
 ' climbs to $68,995 for the manual coupe, and $72,',
 '-year-old protagonist who, “...thinks about how lucky she',
 ' of these immigrants’ lives and sensitively expose the hopelessness of their predicament',
 'Mendenhall will get help from Ryan Williams and company in the backfield but',
 ' just 2 months\n→ The number of coronavirus cases in India has risen',
 'terson eventually received an exculpatory DNA lab report, through his own Free

In [12]:
task.dit.tokenizer.batch_decode(out)

[' want to continue to put out good work.\nHHNM: What plans does<|think|><|think|>I: I I: I I:: I: try try to to:: I I:: I I try try: I: I try: I:: I: try I: I I to try:I think\n? I:::: think I I think: I think. I I',
 '-lap contest, one where a late scuffle by Sebastian Vettel saw the<|think|><|think|> back the of front. back the and of back the front the back in the the front of back the and in front. �The the all in the of the, the the, in the all the, the when of the,,, of the the and when, –, the when�’�',
 ' the Colorado School of Public Health, will be the first director of the Center for<|think|><|think|>, and prevention management prevention, preventive and and prevention management prevention, of treatment, treatment management and treatment prevention of and, and treatment clinical, clinical management and medical of treatment, treatment of management, of and and treatmentThe�\n�. us all of in the and, hospital the to go treating the for all',
 ' exceptional results, as it i

In [15]:
gen_suffix_str_joint[0]

' have you for��\n��’’��’� ’�’’�’����’�,‘‘��’������’�’���’����'

In [ ]:
ppl_joint = self.eval_ppl(
    batch["prefix_str"],
    gen_suffix_str_joint,
    device=batch["input_ids_dec"].device,
)